# Screen: >100% return in 1 calendar year

EQ stocks with total return **> 100%** over **one calendar year**:

- `start_date` = last bar on or before `end_date − 1 year`
- `end_date` = latest session (or `AS_OF`)
- `window_days` must be in **[300, 400]** (drops gappy 252-row windows)

Liquidity at `start_date`: avg volume & turnover over 5/10/15/20/25/30/45/60/75/90 sessions.

Logic lives in `scripts/screen_100pct_1y.py`. Set `QUANT_DATA_DIR` if needed (`quant.config`).

In [ ]:
from pathlib import Path
import sys
import subprocess
import importlib.util

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src" / "quant").is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

for pkg in ("pandas", "pyarrow", "matplotlib"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import pandas as pd

from quant.config import DATA_DIR, PARQUET_DATA_DIR

_script = REPO_ROOT / "scripts" / "screen_100pct_1y.py"
_spec = importlib.util.spec_from_file_location("screen_100pct_1y", _script)
assert _spec and _spec.loader
screen = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(screen)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)

print("DATA_DIR:", DATA_DIR)
print("PARQUET_DATA_DIR:", PARQUET_DATA_DIR, "exists:", PARQUET_DATA_DIR.exists())

In [ ]:
# Optional overrides (defaults in screen_100pct_1y.py)
screen.AS_OF = None  # e.g. pd.Timestamp("2025-11-21")
screen.MIN_RETURN = 1.0
screen.MIN_PRICE = 5.0
screen.EQ_ONLY = True

panel = screen.load_panel()
print("panel:", panel.shape, panel["Date"].min(), "→", panel["Date"].max())

winners = screen.stocks_over_100pct_1y(panel, as_of=screen.AS_OF)
ever = screen.symbols_ever_100pct_in_rolling_year(panel)

print(f"Latest screen: {len(winners)} symbols")
if len(winners):
    display(winners.head(30))
else:
    print("No matches at latest date.")

In [ ]:
screen.export_csv(winners, ever)
print("Wrote:", screen.CSV_LATEST)
print("Wrote:", screen.CSV_EVER_BEST)

## Volume & close around the 1y window

Plot ±6 **calendar** months around `start_date` … `end_date`.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

SYMBOL = "RELIANCE"  # change me

row = winners.loc[winners["Symbol"].eq(SYMBOL.upper().strip())]
if row.empty:
    raise ValueError(f"{SYMBOL} not in winners — pick a symbol from the table above.")
row = row.iloc[0]

start_date = pd.Timestamp(row["start_date"])
end_date = pd.Timestamp(row["end_date"])
plot_start = start_date - pd.DateOffset(months=6)
plot_end = end_date + pd.DateOffset(months=6)
sym = SYMBOL.upper().strip()

bars = panel.copy()
bars["Date"] = pd.to_datetime(bars["Date"])
bars["Symbol"] = bars["Symbol"].astype(str).str.upper().str.strip()
bars["Volume"] = pd.to_numeric(bars["Volume"], errors="coerce")
bars["Close"] = pd.to_numeric(bars["Close"], errors="coerce")

sub = bars.loc[
    (bars["Symbol"] == sym) & (bars["Date"] >= plot_start) & (bars["Date"] <= plot_end),
    ["Date", "Volume", "Close"],
].dropna(subset=["Date"]).sort_values("Date")

if sub.empty:
    raise ValueError(f"No bars for {sym} in [{plot_start.date()} → {plot_end.date()}]")

ret_pct = float(row["ret_1y_pct"]) if "ret_1y_pct" in row.index else float(row["ret_1y"]) * 100

fig, ax_vol = plt.subplots(figsize=(14, 5))
ax_price = ax_vol.twinx()

ax_vol.bar(sub["Date"], sub["Volume"], width=0.8, color="steelblue", alpha=0.45, label="Volume")
ax_price.plot(sub["Date"], sub["Close"], color="darkorange", lw=1.8, label="Close")

for d, color in ((start_date, "green"), (end_date, "red")):
    ax_vol.axvline(d, color=color, ls="--", lw=1.2)
    ax_price.axvline(d, color=color, ls="--", lw=1.2)

ax_vol.set_ylabel("Volume", color="steelblue")
ax_price.set_ylabel("Close", color="darkorange")
ax_vol.set_title(
    f"{sym} | {plot_start.date()} → {plot_end.date()}\n"
    f"1y window {start_date.date()} → {end_date.date()} "
    f"({row['start_price']:.2f} → {row['end_price']:.2f}, {ret_pct:.1f}%, window_days={int(row['window_days'])})"
)
ax_vol.xaxis.set_major_locator(mdates.MonthLocator())
ax_vol.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(ax_vol.xaxis.get_majorticklabels(), rotation=45, ha="right")
ax_vol.grid(axis="y", alpha=0.3)

h1, l1 = ax_vol.get_legend_handles_labels()
h2, l2 = ax_price.get_legend_handles_labels()
ax_vol.legend(h1 + h2, l1 + l2, loc="upper left")
fig.tight_layout()
plt.show()